# 04.01 · Business Intelligence desde cero

No empiezas con una gráfica: **empiezas con una decisión**. Caso ficticio: una comercializadora necesita priorizar mercados y productos.

**Producto final**: 4 KPIs, 3 visualizaciones y recomendación escrita con supuestos.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
root = Path('../..') if Path('../../00 Fundamentos de Datos - Excel y SQL/datos/ventas.csv').exists() else Path('.')
data=root/'00 Fundamentos de Datos - Excel y SQL'/'datos'
v=pd.read_csv(data/'ventas.csv',parse_dates=['fecha'])
c=pd.read_csv(data/'clientes.csv')
p=pd.read_csv(data/'productos.csv')
df=v.merge(c,on='cliente_id',validate='many_to_one').merge(p,on='producto_id',validate='many_to_one')
df['ingreso']=df.unidades*df.precio_unitario
df['costo']=df.unidades*df.costo_unitario
df['margen']=df.ingreso-df.costo
assert len(df)==len(v)
df.head()

## Contrato de KPI
- Ingresos: suma de cantidad * precio
- Margen bruto: ingresos - costo variable
- Margen porcentual: margen agregado / ingreso agregado
- Clientes activos: clientes con al menos una operación en este conjunto

**Evita llamar utilidad neta al margen bruto.**

In [ ]:
ingresos=df.ingreso.sum();margen=df.margen.sum()
kpis={'ingresos':ingresos,'margen_bruto':margen,'margen_pct':100*margen/ingresos,'clientes_activos':df.cliente_id.nunique()}
pd.Series(kpis)

In [ ]:
por_pais=df.groupby('pais').agg(ingreso=('ingreso','sum'),margen=('margen','sum'))
por_producto=df.groupby('producto').agg(ingreso=('ingreso','sum'),margen=('margen','sum'))
mensual=df.set_index('fecha').resample('MS')['ingreso'].sum()
por_pais.plot.bar(title='Ingresos y margen por mercado');plt.tight_layout();plt.show()
por_producto['margen'].sort_values().plot.barh(title='Margen bruto por producto');plt.tight_layout();plt.show()
mensual.plot(title='Ingreso mensual',marker='o');plt.tight_layout();plt.show()

## Diseña el dashboard
¿Cuál es la pregunta de cada visualización? Si ninguna decisión cambia al verla, considera eliminarla. ¿Qué segmento puede ocultarse al agrupar por país?

In [ ]:
participacion=df.groupby('cliente').ingreso.sum().sort_values(ascending=False)
resultado=pd.DataFrame({'ingresos':participacion,'participacion_pct':100*participacion/participacion.sum(),'acumulado_pct':100*participacion.cumsum()/participacion.sum()})
resultado

## Reto extendido
1. Propón un nuevo indicador y define unidad, denominador, granularidad y frecuencia.
2. Identifica qué dimensión sesga más la lectura.
3. Presenta recomendación y una hipótesis falsable; describe qué datos pedirías para contrastarla.
4. Reproduce el tablero en Streamlit y verifica totales frente a Excel/PostgreSQL.
5. Indica limitaciones de la muestra y por qué no puedes inferir causalidad.